In [1]:
# Import general packages
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
from itertools import product
from tqdm import tqdm

# Import Agent-Environment packages
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_ordered_index import AddReactionByOrderedIndex
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex

# Import Interface packages
from RL4CRN.env2agent_interface.explicit_observer import ExplicitObserver
from RL4CRN.env2agent_interface.explicit_tensorizer import ExplicitTensorizer
from RL4CRN.agent2env_interface.library_actuator import LibraryActuator
from RL4CRN.agent2env_interface.iocrn_stepper import IOCRNStepper

# Import CRN packages
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library

# Import Reward packages
from RL4CRN.rewards.stochastic import dynamic_tracking_error_SSA, robust_tracking_loss_SSA

In [2]:
# Set the logger to use Comet
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_key = "vhIR3uyqsKyU4L7SA8fLCfTSC"
task_name = "stochastic_s3_r5_REINFORCE_with_CV"
logger = CometLogger(
    api_key=api_key,
    project=task_name,        
    workspace="redsnic", 
    name=f'{task_name}_{timestamp}',
)
logger = logger.experiment

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/redsnic/stochastic-s3-r5-reinforce-with-cv/ba28f181df8e4fb589c6e7efc65d8b3e



In [3]:
# Construct the template CRN
scale = 3.0
r1 = MassAction(reactant_labels=[], product_labels=['Z_1'], input_channels=['u_1'], params=[scale], params_controllability=[True])
r2 = MassAction(reactant_labels=['X_1'], product_labels=[], input_channels=['u_2'], params=[1.], params_controllability=[True])
crn_template = IOCRN([r1, r2], output_labels=['X_1'])
crn_template.compile()
p = crn_template.num_inputs # Number of inputs of the IOCRNs
print("Template CRN:")
print(crn_template)

# Construct the library of possible reactions
species_labels = ['X_1', 'Z_1', 'Z_2'] + ['Z_3']
library = construct_mass_action_library(species_labels=species_labels, order=2)
crn_template.set_library_context(library)
M = len(library.reactions) # Number of possible reactions
K = library.get_num_parameters() # Total number of parameters in all the reactions of the library
print("Library of possible reactions:")
print(library)
print("------------------------------------------------")

Template CRN:
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1'] 
Output Species: ['X_1'] 
∅ ----> Z_1;  [MAK(3.0, u_1)]
X_1 ----> ∅;  [MAK(1.0, u_2)]
Library of possible reactions:
Number of reactions: 211
R0: ∅ ----> ∅;  [MAK(None)]
R1: ∅ ----> X_1;  [MAK(None)]
R2: ∅ ----> Z_1;  [MAK(None)]
R3: ∅ ----> Z_2;  [MAK(None)]
R4: ∅ ----> Z_3;  [MAK(None)]
R5: ∅ ----> X_1 + X_1;  [MAK(None)]
R6: ∅ ----> X_1 + Z_1;  [MAK(None)]
R7: ∅ ----> X_1 + Z_2;  [MAK(None)]
R8: ∅ ----> X_1 + Z_3;  [MAK(None)]
R9: ∅ ----> Z_1 + Z_1;  [MAK(None)]
R10: ∅ ----> Z_1 + Z_2;  [MAK(None)]
R11: ∅ ----> Z_1 + Z_3;  [MAK(None)]
R12: ∅ ----> Z_2 + Z_2;  [MAK(None)]
R13: ∅ ----> Z_2 + Z_3;  [MAK(None)]
R14: ∅ ----> Z_3 + Z_3;  [MAK(None)]
R15: X_1 ----> ∅;  [MAK(None)]
R16: X_1 ----> Z_1;  [MAK(None)]
R17: X_1 ----> Z_2;  [MAK(None)]
R18: X_1 ----> Z_3;  [MAK(None)]
R19: X_1 ----> X_1 + X_1;  [MAK(None)]
R20: X_1 ----> X_1 + Z_1;  [MAK(None)]
R21: X_1 ----> X_1 + Z_2;  [MAK(None)]
R22: X_1 ----> X_1 + Z_3;  [MAK(None

In [5]:
# Device
import os
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'Number of CPUs available: {os.cpu_count()}') 

Using device: cuda
Number of CPUs available: 128


In [6]:
# Flags and filenames
save_flag = True                                                # Save the agent checkpoint
load_flag = False                                               # Load the agent checkpoint 
train_flag = True                                               # Train the agent
save_sheet_flag = True                                          # Save the configuration to an Excel sheet

save_filename = timestamp + '.pth'                              # Filename for saving the agent checkpoint
load_filename = ''                                              # Filename for loading the agent checkpoint
file_name = f"{task_name}.xlsx"             # Filename for saving the Excel sheet

In [7]:
# Hyperparameters
max_added_reactions = 6                             # Maximum number of reactions
N_CPUs = 4 # os.cpu_count()                         # Number of CPUs          
N = 160                                             # Number of samples (batch size)    
width = 1024                                        # Width of the neural networks  
depth = 5                                           # Depth of the neural networks 
deep_layer_size = 1024*10                           # Size of the deep layer encoding the CRNs
allow_input_influence = False                       # Allow input influence in the policy
learning_rate = 1e-4                                # Learning rate for the optimizer 
hall_of_fame_size = 30                              # Size of the hall of fame  
entropy_scheduler = {                               # Entropy scheduler parameters 
    'entropy_weight': 5e-3, 
    'topk_entropy_weight' : 1.0,
    'remainder_entropy_weight' : 1.0,
    'entropy_update_coefficient': 1, 
    'entropy_schedule': 1000, 
    'minimum_entropy_weight': 0.0
}
entropy_weights_per_head = {'structure': 3.0, 'continuous': 1.0, 'discrete': 0.0, 'input_influence': 0.0} 
structure_head_temperature = {"target_entropy_ratio_to_max": np.log(5)/np.log(M), "initial_temperature": 1.0, "rate": 0.0, "current_temperature": 1.0}
risk_scheduler = {                                  # Risk scheduler parameters
    'risk': 0.90, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 1000
}
epoch_num = 600                                     # Number of epochs for training
render_schedule = 10                                # Render every # of epochs
render_mode = {                                     # Mode of the experiment
    'style': 'logger', 
    'task': 'SSA_transients', 
    'format': 'image',
    'topology': True,
    'bounds': [25]
}
# Ordering specific parameters
ordering_parameters = {
    'enforce_ordering': False,
    'constraint_weight' : float('inf')
}
# SIL settings
sil_settings = {
    'sil_loss_weight': 1.0,
    'sil_use_adaptive_baseline': False,
    'sil_baseline_annealing_rate': 0.95
}

render_n_best = 10                                                     # Number of best CRNs to plot responses for
render_disregard_percentage = risk_scheduler['risk']                  # Percentage of worst CRNs to disregard in the responses plotting

# Parameter distribution for the reactions added by the agent
continuous_distribution = {"type": 'lognormal_1D'}

# Time horizon for the simulation
t_f = 100                                           # Final time for the simulation
N_t = 100                                          # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Construct the IOCRN inputs
nums = [1, 2, 3]
disturbances = [0.5, 1, 1.5]
u_list = [np.array(u) for u in product(nums, disturbances)] # list of input combinations, each input is a numpy array of shape (p,)

print(u_list)

# Construct the reference setpoints
r_list = [np.array([u[0]])*scale for u in u_list]
print(r_list)   

# Construct the IOCRN initial conditions
ic = IC(names=species_labels, values=[[0.0, 0.0, 0.0, 0.0]]) 

# Construct the weights for the performance metric
w = np.ones(N_t)
w[(len(w)//5)*4:] = w[(len(w)//5)*4:]*2
w[:2*(len(w)//5)] = w[:2*(len(w)//5)]*0.
w = w[np.newaxis, :]

# Construct the compute reward routine
def compute_reward(state):
    x0_list = ic.get_ic(state)
    # return dynamic_tracking_error_SSA(state, u_list, x0_list, time_horizon, r_list, w, norm=2, LARGE_NUMBER=1e6, max_threads=1024, n_trajectories=1024)
    return robust_tracking_loss_SSA(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e3, LARGE_PENALTY=100, max_threads=1024, n_trajectories=1024, relative=True, cv_weight=1., rpa_weight=3.)

[array([1. , 0.5]), array([1, 1]), array([1. , 1.5]), array([2. , 0.5]), array([2, 1]), array([2. , 1.5]), array([3. , 0.5]), array([3, 1]), array([3. , 1.5])]
[array([3.]), array([3.]), array([3.]), array([6.]), array([6.]), array([6.]), array([9.]), array([9.]), array([9.])]


In [8]:
if save_sheet_flag:
    sheet_name = "Data"
    headers = [
        "Timestamp", "URL",
        "Epochs Completed", "Successful", "Saved", "Comments",
        "Learning Rate", "Epochs #",
        "(m, n, p, N)",
        "NN Depth", "NN Width", "Deep Layer Size", "CPUs #",
        "Entropy Scheduler",
        "Risk Scheduler",
        "Render Schedule", "HoF Size",
        "Simulation Time", "Time Steps #",
        "Initial Conditions #", "Input Scenarios#",
        "Continuous Distribution", "Entropy Weights per Head",
        "Structure Head Temperature",
        "Ordering Enforced",
        "SIL Settings"
    ]

    data_row = [
        timestamp, logger.url,
        None, None, None, None,
        learning_rate, epoch_num,
        str((max_added_reactions, len(species_labels), p, N)),
        depth, width, deep_layer_size, N_CPUs,
        str(entropy_scheduler),
        str(risk_scheduler),
        render_schedule, hall_of_fame_size,
        t_f, N_t, len(ic.values), len(u_list),
        str(continuous_distribution), str(entropy_weights_per_head),
        str(structure_head_temperature),
        f"Yes: {ordering_parameters['constraint_weight']}" if ordering_parameters['enforce_ordering'] else "No",
        str(sil_settings)
    ]

    if os.path.exists(file_name):
        wb = load_workbook(file_name)
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
        else:
            ws = wb.create_sheet(sheet_name)
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = sheet_name

    # Write headers if sheet is empty
    if ws.max_row == 1 and ws.max_column == 1 and ws.cell(row=1, column=1).value is None:
        for col, header in enumerate(headers, start=1):
            ws.cell(row=1, column=col, value=header)

    # Append experiment as next row
    next_row = ws.max_row + 1
    for col, value in enumerate(data_row, start=1):
        ws.cell(row=next_row, column=col, value=value)

    # Freeze header row and add filter
    ws.freeze_panes = "B1" 
    ws.auto_filter.ref = ws.dimensions

    # === Auto-fit column widths (except URL column) ===
    # URL column is column 2 (B), we leave its width unchanged.
    url_col_index = 2

    for col in range(1, ws.max_column + 1):
        if col == url_col_index:
            continue  # keep URL column width as-is

        max_length = 0
        for row in range(1, ws.max_row + 1):
            cell = ws.cell(row=row, column=col)
            value = cell.value
            if value is not None:
                # Convert to string to measure length
                length = len(str(value))
                if length > max_length:
                    max_length = length

        # Some padding so text isn't touching the cell border
        adjusted_width = max_length + 2 if max_length > 0 else 10
        col_letter = get_column_letter(col)
        ws.column_dimensions[col_letter].width = adjusted_width

    wb.save(file_name)
    print(f"New experiment data saved in row {next_row} of '{file_name}'.")

New experiment data saved in row 13 of 'stochastic_s3_r5_REINFORCE_with_CV.xlsx'.


In [9]:
# Construct parallel environments
crn_0 = crn_template.clone()
mult_env = ParallelEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, N_CPUs=N_CPUs, logger=logger)
# mult_env = SerialEnvironments([Environment(crn_0, max_added_reactions, logger=logger, logger_schedule=1) for _ in range(N)], hall_of_fame_size=hall_of_fame_size, logger=logger)

In [10]:
# Construct the policy
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
masks = {"continuous": library.get_parameter_mask(mode="continuous"), "discrete": library.get_parameter_mask(mode="discrete"), "logit": library.get_logit_mask()}
policy = AddReactionByOrderedIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, target_set_size=crn_template.num_reactions+max_added_reactions, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head,
                                    combinatorial_bias_enabled=ordering_parameters["enforce_ordering"], constraint_strength=ordering_parameters["constraint_weight"])

if ordering_parameters["enforce_ordering"]:
    policy = AddReactionByOrderedIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, target_set_size=crn_template.num_reactions+max_added_reactions, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head,
                                        combinatorial_bias_enabled=ordering_parameters["enforce_ordering"], constraint_strength=ordering_parameters["constraint_weight"])
else:
    policy = AddReactionByIndex(M, K, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, allow_input_influence=False, masks=masks, device=device, continuous_distribution=continuous_distribution, entropy_weights_per_head=entropy_weights_per_head)

# Construct the agent
agent = REINFORCEAgent(policy, allow_input_influence=False, logger=logger, learning_rate=learning_rate, entropy_scheduler=entropy_scheduler, risk_scheduler=risk_scheduler, sil_settings=sil_settings, device=device)
if load_flag:
    agent.policy.load_state_dict(torch.load(load_filename+'.pth', map_location=device))

In [11]:
# Construct the interfaces
observer = ExplicitObserver(reaction_library=library, allow_input_observation=allow_input_influence)
tensorizer = ExplicitTensorizer(device=device)
actuator = LibraryActuator(reaction_library=library)
stepper = IOCRNStepper()

In [ ]:
# Training Loop   

if train_flag:
    agent.policy.train()
    for i in tqdm(range(epoch_num)):
        mult_env.reset()
        for j in range(max_added_reactions):
            observations = mult_env.observe(observer, tensorizer)
            actions, raw_actions = agent.act(observations, actuator)
            out = mult_env.step(actions, stepper, raw_actions=raw_actions)
        rewards = mult_env.get_reward(compute_reward)

        # count how many environments were successful (i.e., did not diverge)
        successful_count = sum(1 for env in mult_env.envs if not env.state.last_task_info.get('has_diverged', False))
        # Log the number of successful environments
        logger.log_metric("Successful Environments (%)", successful_count/N, step=i)
        print(f"Info: in epoch {i}: successful simulation rate {successful_count/N} ({successful_count}/{N})")

        agent.update(rewards, step_iteration=i, hof=mult_env.hall_of_fame, observer=observer, tensorizer=tensorizer, stepper=stepper, use_sil=True, sil_weighting_scheme='uniform', sil_batch_size=None)
        if i % render_schedule == 0:
            mult_env.render(rewards, n_best=render_n_best, disregarded_percentage=render_disregard_percentage, mode=render_mode)

  0%|          | 0/600 [00:00<?, ?it/s]

In [ ]:
hall_of_fame_crns = [env.state for env in mult_env.hall_of_fame]
if save_flag:
    if not os.path.exists('models'):
        os.makedirs('models')
    if not os.path.exists('hof'):
        os.makedirs('hof')
    torch.save(agent.policy.state_dict(), 'models/' + save_filename)
    torch.save(hall_of_fame_crns, 'hof/hall_of_fame_' + save_filename)

In [ ]:
[f"Loss::: {str(c.last_task_info['reward'])} CRN::: {str(c)}" for c  in hall_of_fame_crns[0:10]]

["Loss::: 0.20703635381413038 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2', 'Z_3'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nZ_1 ----> X_1 + Z_2;  [MAK(5.079129219055176)]\nZ_2 ----> X_1;  [MAK(1.2960373163223267)]\nX_1 + X_1 ----> X_1;  [MAK(7.721121311187744)]\nZ_2 + Z_2 ----> Z_1 + Z_1;  [MAK(3.6516335010528564)]\nZ_2 + Z_2 ----> Z_1 + Z_2;  [MAK(5.114426612854004)]\nZ_2 + Z_3 ----> Z_2;  [MAK(2.259320020675659)]",
 "Loss::: 0.20813192485381363 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2', 'Z_3'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nZ_1 ----> X_1 + Z_2;  [MAK(5.4630818367004395)]\nZ_2 ----> X_1;  [MAK(1.6133368015289307)]\nZ_3 ----> X_1 + Z_2;  [MAK(8.143619537353516)]\nZ_3 ----> Z_2 + Z_2;  [MAK(7.920536994934082)]\nX_1 + X_1 ----> X_1;  [MAK(8.54217529296875)]\nZ_2 + Z_2 ----> Z_1 + Z_1;  [MAK(10.482694625854492)]",
 "Loss::: 0.20846104492058448 CR

In [ ]:
str(library)

'Number of reactions: 211\nR0: ∅ ----> ∅;  [MAK(None)]\nR1: ∅ ----> X_1;  [MAK(None)]\nR2: ∅ ----> Z_1;  [MAK(None)]\nR3: ∅ ----> Z_2;  [MAK(None)]\nR4: ∅ ----> Z_3;  [MAK(None)]\nR5: ∅ ----> X_1 + X_1;  [MAK(None)]\nR6: ∅ ----> X_1 + Z_1;  [MAK(None)]\nR7: ∅ ----> X_1 + Z_2;  [MAK(None)]\nR8: ∅ ----> X_1 + Z_3;  [MAK(None)]\nR9: ∅ ----> Z_1 + Z_1;  [MAK(None)]\nR10: ∅ ----> Z_1 + Z_2;  [MAK(None)]\nR11: ∅ ----> Z_1 + Z_3;  [MAK(None)]\nR12: ∅ ----> Z_2 + Z_2;  [MAK(None)]\nR13: ∅ ----> Z_2 + Z_3;  [MAK(None)]\nR14: ∅ ----> Z_3 + Z_3;  [MAK(None)]\nR15: X_1 ----> ∅;  [MAK(None)]\nR16: X_1 ----> Z_1;  [MAK(None)]\nR17: X_1 ----> Z_2;  [MAK(None)]\nR18: X_1 ----> Z_3;  [MAK(None)]\nR19: X_1 ----> X_1 + X_1;  [MAK(None)]\nR20: X_1 ----> X_1 + Z_1;  [MAK(None)]\nR21: X_1 ----> X_1 + Z_2;  [MAK(None)]\nR22: X_1 ----> X_1 + Z_3;  [MAK(None)]\nR23: X_1 ----> Z_1 + Z_1;  [MAK(None)]\nR24: X_1 ----> Z_1 + Z_2;  [MAK(None)]\nR25: X_1 ----> Z_1 + Z_3;  [MAK(None)]\nR26: X_1 ----> Z_2 + Z_2;  [MAK